# 3 · Invoice Extractor Agent
## Nested Schemas, Validators, and the Contract Pattern
⏱ ~15 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent-use-cases/blob/main/examples/3-invoice-extractor/invoice_extractor_workbook.ipynb)

Finance teams at most companies still copy invoice data by hand — vendor name, line items, totals — into spreadsheets or ERP systems. It's slow, error-prone, and completely automatable.

This example builds an agent that reads raw invoice text and returns a fully structured `Invoice` object: vendor, date, every line item, subtotal, tax, and total. A Pydantic validator enforces that the extracted total is always positive — if the model gets it wrong, it retries automatically.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | The business problem: manual invoice processing |
| 2 | Nested schemas: `Invoice` containing `LineItem` objects |
| 3 | Field validators: the model retries if validation fails |
| 4 | Running the extractor |
| 5 | Handling real-world invoice variation |
| ★ | Exercise: add a payment terms field + test edge cases |

---

### Prerequisites
- Python 3.10+
- `OPENAI_API_KEY` in `.env` or Colab Secrets

### Key References
> Pydantic validators: https://docs.pydantic.dev/latest/concepts/validators/  
> LangChain structured output: https://python.langchain.com/docs/concepts/structured_outputs/

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "pydantic", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local environment — skipping install.")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Loaded API key from Colab Secrets.")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded API key from .env file.")

key = os.environ.get("OPENAI_API_KEY", "")
if key and key.startswith("sk-"):
    print("✓ OPENAI_API_KEY is set and looks valid.")
else:
    print("✗ OPENAI_API_KEY missing or invalid.")
    print("  Colab: Secrets panel (key icon) → Add secret 'OPENAI_API_KEY'")
    print("  Local: create a .env file with OPENAI_API_KEY=sk-...")

---

## Part 1 — The Business Problem

---

A typical invoice lands in a finance team's inbox as a PDF attachment or email body. Processing it means:

1. Open the invoice
2. Find the vendor name
3. Find the invoice number and date
4. Copy each line item (description, quantity, unit price, total)
5. Verify the subtotal, tax, and grand total
6. Enter everything into the accounting system

For a company receiving 200 invoices a month, this is 800+ hours a year of manual data entry.

An extraction agent does steps 1–5 in under 2 seconds. The finance team only touches step 6 — and only when something is flagged for review.

```
raw invoice text  →  Invoice extractor agent  →  validated Invoice object
                                                   ↓
                                              accounting system API
```

---

## Part 2 — Nested Schemas

---

An invoice isn't a flat list of fields — it contains a **list of line items**. Pydantic handles this natively with nested models:

```
Invoice
  ├── vendor        str
  ├── invoice_number str
  ├── date          str
  ├── subtotal      float
  ├── tax           float
  ├── total_amount  float  ← validated: must be > 0
  └── line_items    List[LineItem]
        ├── description  str
        ├── quantity     int
        ├── unit_price   float
        └── total        float
```

LangChain sends the *entire* nested schema to the model. The model must populate every level — including each `LineItem` in the list.

In [ ]:
from typing import List
from pydantic import BaseModel, Field, field_validator
import json


class LineItem(BaseModel):
    description: str = Field(description="Name or description of the product or service")
    quantity: int = Field(description="Number of units")
    unit_price: float = Field(description="Price per single unit in USD")
    total: float = Field(description="quantity × unit_price for this line item")


class Invoice(BaseModel):
    vendor: str = Field(description="Full name of the vendor or supplier")
    invoice_number: str = Field(description="Invoice reference number, e.g. INV-2024-0892")
    date: str = Field(description="Invoice date in ISO format: YYYY-MM-DD")
    subtotal: float = Field(description="Sum of all line item totals before tax, in USD")
    tax: float = Field(description="Tax amount in USD")
    total_amount: float = Field(description="Final amount due: subtotal + tax, in USD")
    line_items: List[LineItem] = Field(description="All individual line items on the invoice")

    @field_validator("total_amount")
    @classmethod
    def must_be_positive(cls, v: float) -> float:
        """Reject any invoice where the total is zero or negative."""
        if v <= 0:
            raise ValueError(f"total_amount must be positive, got {v}")
        return v


print("Invoice JSON Schema:")
print(json.dumps(Invoice.model_json_schema(), indent=2))

---

## Part 3 — Field Validators: Automatic Retry on Failure

---

The `@field_validator("total_amount")` decorator runs after the model returns its response. If `total_amount <= 0`, Pydantic raises a `ValidationError`.

When you use `with_structured_output()`, LangChain catches this error and **automatically retries the API call** — the model gets another chance to produce a valid response.

```
invoice text  →  LLM call  →  Pydantic validates
                               ↓ if total_amount <= 0: ValidationError
                               ↓ LangChain retries automatically
                               ↓ LLM call #2
                               ↓ Pydantic validates again
                               ↓ if valid: return Invoice object
```

This is the **contract pattern**: the schema is not just a shape, it's a contract with enforceable rules. The agent cannot return a broken invoice — it either produces a valid one or fails with a clear error.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
extractor = llm.with_structured_output(Invoice)

SAMPLE_INVOICE = """
INVOICE #INV-2024-0892
Vendor: CloudPeak Software Ltd
Date: 2024-05-15

Line Items:
  Annual SaaS License (5 seats)    x1   $2,400.00
  Premium Support Package           x1     $600.00
  Onboarding & Setup                x3     $150.00  =  $450.00

Subtotal:  $3,450.00
Tax (15%):   $517.50
TOTAL DUE: $3,967.50

Payment due within 30 days.
"""

invoice = extractor.invoke(SAMPLE_INVOICE)

print(f"Return type: {type(invoice).__name__}")
print()
print(f"Vendor:   {invoice.vendor}")
print(f"Invoice#: {invoice.invoice_number}")
print(f"Date:     {invoice.date}")
print(f"Total:    ${invoice.total_amount:,.2f}")
print(f"\nLine items ({len(invoice.line_items)}):")
for item in invoice.line_items:
    print(f"  - {item.description}")
    print(f"    {item.quantity} × ${item.unit_price:,.2f} = ${item.total:,.2f}")

In [ ]:
# Serialize to dict for downstream use (e.g. API call, database write)
import json

invoice_dict = invoice.model_dump()
print("Invoice as dict (ready for any downstream system):")
print(json.dumps(invoice_dict, indent=2))

---

## Part 4 — Handling Invoice Variation

---

Real invoices don't all look the same. Let's test a second invoice in a completely different format.

In [ ]:
INVOICE_2 = """
TAX INVOICE
From: Apex Cloud Consulting
To: Meridian Corp
Invoice No: APC-2024-112  |  Date: 15 June 2024

Services rendered:
  System Architecture Review    1 day    @ $3,500/day     $3,500
  Implementation Support        3 days   @ $2,200/day     $6,600
  Documentation                 0.5 day  @ $1,800/day       $900

Subtotal ........................ $11,000.00
GST (10%) ....................... $1,100.00
TOTAL ........................... $12,100.00
"""

invoice2 = extractor.invoke(INVOICE_2)

print(f"Vendor:   {invoice2.vendor}")
print(f"Invoice#: {invoice2.invoice_number}")
print(f"Date:     {invoice2.date}")
print(f"Total:    ${invoice2.total_amount:,.2f}")
print(f"\nLine items ({len(invoice2.line_items)}):")
for item in invoice2.line_items:
    print(f"  - {item.description}: {item.quantity} × ${item.unit_price:,.2f} = ${item.total:,.2f}")

---

## Exercise — Add Payment Terms and Test an Edge Case

**Part A:** Add a `payment_terms` field to the `Invoice` schema.  
It should capture things like `"Net 30"`, `"Due on receipt"`, `"Net 60"`.  
Use `Optional[str]` since not all invoices include payment terms.

**Part B:** Test the validator by trying to extract from a broken invoice (missing total).  
Observe the error — this is what your error handling code should catch in production.

**Questions to consider:**
1. Does the model handle `None` correctly for invoices without payment terms?
2. What does the `ValidationError` look like when `total_amount` is wrong?

In [ ]:
# ── Exercise Starter ────────────────────────────────────────────────────────
from typing import List, Optional
from pydantic import BaseModel, Field, field_validator

class LineItemV2(BaseModel):
    description: str = Field(description="Name or description of the product or service")
    quantity: int = Field(description="Number of units")
    unit_price: float = Field(description="Price per single unit in USD")
    total: float = Field(description="quantity × unit_price for this line item")

class InvoiceV2(BaseModel):
    vendor: str = Field(description="Full name of the vendor or supplier")
    invoice_number: str = Field(description="Invoice reference number")
    date: str = Field(description="Invoice date in ISO format: YYYY-MM-DD")
    subtotal: float = Field(description="Sum of all line item totals before tax")
    tax: float = Field(description="Tax amount in USD")
    total_amount: float = Field(description="Final amount due: subtotal + tax")
    line_items: List[LineItemV2] = Field(description="All line items on the invoice")
    # TODO: add payment_terms: Optional[str] = Field(default=None, description="...")

    @field_validator("total_amount")
    @classmethod
    def must_be_positive(cls, v):
        if v <= 0:
            raise ValueError(f"total_amount must be positive, got {v}")
        return v

# TODO: test on SAMPLE_INVOICE — does payment_terms extract correctly?
# TODO: test on INVOICE_2 — does it return None for payment_terms?

In [ ]:
# ── Answer Key ──────────────────────────────────────────────────────────────

class InvoiceV2Answer(BaseModel):
    vendor: str = Field(description="Full name of the vendor or supplier")
    invoice_number: str = Field(description="Invoice reference number")
    date: str = Field(description="Invoice date in ISO format: YYYY-MM-DD")
    subtotal: float = Field(description="Sum of all line item totals before tax")
    tax: float = Field(description="Tax amount in USD")
    total_amount: float = Field(description="Final amount due: subtotal + tax")
    line_items: List[LineItem] = Field(description="All line items on the invoice")
    payment_terms: Optional[str] = Field(
        default=None,
        description="Payment terms, e.g. 'Net 30', 'Due on receipt'. Null if not stated."
    )

    @field_validator("total_amount")
    @classmethod
    def must_be_positive(cls, v):
        if v <= 0:
            raise ValueError(f"total_amount must be positive, got {v}")
        return v

extractor_v2 = llm.with_structured_output(InvoiceV2Answer)

inv1 = extractor_v2.invoke(SAMPLE_INVOICE)
inv2 = extractor_v2.invoke(INVOICE_2)

print(f"Invoice 1 payment_terms: {inv1.payment_terms!r}")
print(f"Invoice 2 payment_terms: {inv2.payment_terms!r}")

---

## What's Next?

- **Example 4 — Lead Qualifier** ([`../4-lead-qualifier/`](../4-lead-qualifier/)): combine structured output with a rubric in the system prompt. The score must cite which criteria were met — no black-box numbers.
- **Example 7 — RFP Parser** ([`../7-rfp-parser/`](../7-rfp-parser/)): apply the same extraction pattern to longer documents — requirements, deadlines, and scoring criteria from 10-page procurement RFPs.

### Further Reading
- Pydantic validators: https://docs.pydantic.dev/latest/concepts/validators/
- Pydantic `Optional` and `None` defaults: https://docs.pydantic.dev/latest/concepts/models/#required-optional-fields
- LangChain structured output retry: https://python.langchain.com/docs/concepts/structured_outputs/

---
*Workshop complete. Next: Example 4 — Lead Qualifier Agent.*